In [5]:
import pandas as pd
import os


c:\Users\nshej\fullstack\EdTech-Platform-Analysis\notebooks
False


In [18]:
df_users=pd.read_csv("../data/raw/users.csv")
df_enrollments=pd.read_csv("../data/raw/enrollments.csv")
df_lessons=pd.read_csv("../data/raw/lesson_events.csv")
df_payments=pd.read_csv("../data/raw/payments.csv")

In [19]:
#checking null values for all the data set 

#1.users
print("users")
df_users.info()
df_users.isnull().sum()

print("enrollments")
#2.enrollments
df_enrollments.info()
df_enrollments.isnull().sum()

print("lessons")
#3.lessons
df_lessons.info()
df_lessons.isnull().sum()

print("payments")
#4.payments
df_payments.info()
df_payments.isnull().sum()

users
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   user_id           100000 non-null  object
 1   signup_date       100000 non-null  object
 2   signup_channel    100000 non-null  object
 3   country           100000 non-null  object
 4   age               100000 non-null  int64 
 5   experiment_group  100000 non-null  object
dtypes: int64(1), object(5)
memory usage: 4.6+ MB
enrollments
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149516 entries, 0 to 149515
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   enrollment_id      149516 non-null  object
 1   user_id            149516 non-null  object
 2   course_id          149516 non-null  object
 3   course_category    149516 non-null  object
 4   enrollment_date    149516 non-null  object
 

payment_id              0
enrollment_id           0
amount                  0
payment_date            0
certification_issued    0
dtype: int64

In [21]:
import re


def audit_datetime_parsing(df, name):
    print(f"\n{name} dtypes")
    print(df.dtypes)

    date_like_columns = [
        column
        for column in df.columns
        if any(keyword in column.lower() for keyword in ["date", "time", "timestamp"])
    ]

    if not date_like_columns:
        print(f"No date/timestamp-like columns found in {name}.")
        return

    print(f"\nDate/timestamp parsing audit for {name}")
    for column in date_like_columns:
        parsed = pd.to_datetime(df[column], errors="coerce")
        parse_failures = parsed.isnull().sum()
        non_null_values = df[column].dropna().astype(str).str.strip()

        print(f"- {column}: {df[column].dtype}, parse failures = {parse_failures}")

        if non_null_values.empty:
            print("  no non-null values to inspect")
            continue

        normalized_patterns = (
            non_null_values
            .str.replace(r"\d", "0", regex=True)
            .str.replace(r"\s+", " ", regex=True)
        )
        pattern_counts = normalized_patterns.value_counts()

        if len(pattern_counts) == 1:
            print(f"  pattern check: consistent ({pattern_counts.index[0]})")
        else:
            print(f"  pattern check: mixed formats detected ({len(pattern_counts)} patterns)")
            for pattern, count in pattern_counts.head(3).items():
                print(f"    - {pattern}: {count}")


audit_datetime_parsing(df_users, "users")
audit_datetime_parsing(df_enrollments, "enrollments")
audit_datetime_parsing(df_lessons, "lessons")
audit_datetime_parsing(df_payments, "payments")


users dtypes
user_id             object
signup_date         object
signup_channel      object
country             object
age                  int64
experiment_group    object
dtype: object

Date/timestamp parsing audit for users
- signup_date: object, parse failures = 0
  pattern check: consistent (0000-00-00)

enrollments dtypes
enrollment_id        object
user_id              object
course_id            object
course_category      object
enrollment_date      object
enrollment_status    object
dtype: object

Date/timestamp parsing audit for enrollments
- enrollment_date: object, parse failures = 0
  pattern check: consistent (0000-00-00 00:00:00)

lessons dtypes
event_id           object
enrollment_id      object
lesson_number       int64
event_type         object
event_timestamp    object
dtype: object

Date/timestamp parsing audit for lessons
- event_timestamp: object, parse failures = 3396
  pattern check: mixed formats detected (4 patterns)
    - 0000-00-00 00:00:00: 3440614
    

In [24]:
def audit_numeric_column(df, name, column, *, invalid_rules=None):
    print(f"\n{name}.{column} summary")
    print(df[column].describe())
    print(f"min = {df[column].min()}, max = {df[column].max()}")

    if invalid_rules:
        for label, mask in invalid_rules.items():
            invalid_count = mask(df[column]).sum()
            print(f"{label}: {invalid_count}")


audit_numeric_column(
    df_users,
    "users",
    "age",
    invalid_rules={
        "negative ages": lambda s: s < 0,
        "age == 999": lambda s: s == 999,
    },
)

audit_numeric_column(
    df_payments,
    "payments",
    "amount",
    invalid_rules={
        "negative amounts": lambda s: s < 0,
        "zero amounts": lambda s: s == 0,
    },
 
)

audit_numeric_column(
    df_lessons,
    "lessons",
    "lesson_number",
    invalid_rules={
        "negative amounts": lambda s: s < 0,
        "zero amounts": lambda s: s == 0,
    },
)


users.age summary
count    100000.000000
mean         28.907700
std          18.404877
min          -5.000000
25%          23.000000
50%          28.000000
75%          34.000000
max         999.000000
Name: age, dtype: float64
min = -5, max = 999
negative ages: 96
age == 999: 30

payments.amount summary
count    59755.000000
mean       111.476445
std         29.463903
min         79.000000
25%         89.000000
50%         99.000000
75%        129.000000
max        169.000000
Name: amount, dtype: float64
min = 79.0, max = 169.0
negative amounts: 0
zero amounts: 0

lessons.lesson_number summary
count    3.444010e+06
mean     4.384396e+00
std      2.538716e+00
min      1.000000e+00
25%      2.000000e+00
50%      4.000000e+00
75%      6.000000e+00
max      1.000000e+01
Name: lesson_number, dtype: float64
min = 1, max = 10
negative amounts: 0
zero amounts: 0


In [25]:
def audit_categorical_column(df, name, column):
    print(f"\n{name}.{column} value counts")
    print(df[column].value_counts(dropna=False))

    non_null_values = df[column].dropna().astype(str).str.strip()
    normalized_values = non_null_values.str.lower()
    unique_raw = sorted(non_null_values.unique())
    unique_normalized = sorted(normalized_values.unique())

    print(f"unique values: {unique_raw}")
    print(f"case-normalized values: {unique_normalized}")

    duplicates_after_normalizing = normalized_values.value_counts()
    suspicious = duplicates_after_normalizing[duplicates_after_normalizing > 1]
    if not suspicious.empty:
        print("possible casing/spacing inconsistencies detected:")
        for normalized_value in suspicious.index:
            raw_variants = sorted(non_null_values[normalized_values == normalized_value].unique())
            if len(raw_variants) > 1:
                print(f"- {column}: {raw_variants}")


audit_categorical_column(df_users, "users", "signup_channel")
audit_categorical_column(df_users, "users", "country")
audit_categorical_column(df_users, "users", "experiment_group")
audit_categorical_column(df_enrollments, "enrollments", "course_category")
audit_categorical_column(df_enrollments, "enrollments", "enrollment_status")
audit_categorical_column(df_lessons, "lessons", "event_type")
audit_categorical_column(df_payments, "payments", "certification_issued")


users.signup_channel value counts
signup_channel
organic     35166
paid_ad     29872
social      20075
referral    14887
Name: count, dtype: int64
unique values: ['organic', 'paid_ad', 'referral', 'social']
case-normalized values: ['organic', 'paid_ad', 'referral', 'social']
possible casing/spacing inconsistencies detected:

users.country value counts
country
US    37877
IN    25917
GB    11165
CA     7891
DE     7101
FR     4002
AU     3076
BR     2971
Name: count, dtype: int64
unique values: ['AU', 'BR', 'CA', 'DE', 'FR', 'GB', 'IN', 'US']
case-normalized values: ['au', 'br', 'ca', 'de', 'fr', 'gb', 'in', 'us']
possible casing/spacing inconsistencies detected:

users.experiment_group value counts
experiment_group
treatment    50102
control      49898
Name: count, dtype: int64
unique values: ['control', 'treatment']
case-normalized values: ['control', 'treatment']
possible casing/spacing inconsistencies detected:

enrollments.course_category value counts
course_category
data_analytic

In [26]:
def audit_referential_integrity(child_df, child_name, fk_column, parent_df, parent_name, pk_column):
    print(f"\nReferential integrity: {child_name}.{fk_column} -> {parent_name}.{pk_column}")
    
    parent_values = set(parent_df[pk_column].dropna().unique())
    child_values = child_df[fk_column].dropna()
    
    orphaned = child_df[~child_df[fk_column].isin(parent_values)]
    orphaned_count = len(orphaned)
    
    print(f"Parent ({parent_name}): {len(parent_values)} unique {pk_column} values")
    print(f"Child ({child_name}): {len(child_df)} rows, {child_df[fk_column].notna().sum()} non-null {fk_column}")
    
    if orphaned_count == 0:
        print("✓ All foreign keys are valid (no orphaned records)")
    else:
        print(f"✗ {orphaned_count} orphaned records found (fk not in parent)")
        print(f"  Sample: {orphaned[fk_column].head().tolist()}")


audit_referential_integrity(
    df_enrollments, "enrollments", "user_id",
    df_users, "users", "user_id"
)

audit_referential_integrity(
    df_lessons, "lesson_events", "enrollment_id",
    df_enrollments, "enrollments", "enrollment_id"
)

audit_referential_integrity(
    df_payments, "payments", "enrollment_id",
    df_enrollments, "enrollments", "enrollment_id"
)


Referential integrity: enrollments.user_id -> users.user_id
Parent (users): 100000 unique user_id values
Child (enrollments): 149516 rows, 149516 non-null user_id
✓ All foreign keys are valid (no orphaned records)

Referential integrity: lesson_events.enrollment_id -> enrollments.enrollment_id
Parent (enrollments): 149316 unique enrollment_id values
Child (lesson_events): 3444010 rows, 3444010 non-null enrollment_id
✓ All foreign keys are valid (no orphaned records)

Referential integrity: payments.enrollment_id -> enrollments.enrollment_id
Parent (enrollments): 149316 unique enrollment_id values
Child (payments): 59755 rows, 59755 non-null enrollment_id
✓ All foreign keys are valid (no orphaned records)
